# 01 Dataset exploration

**Objective.** Describe the development data the experiments will use: the score distribution (on the 0.2 grid), how it differs between the frozen train / validation / calibration partitions, and (opt-in) the integrity and intensity properties of the images.

**Scope.** Only the 1000 development images and their labels. The official test set is neither opened nor analysed. Nothing is cleaned or removed: problematic cases are counted and listed, never dropped.

**Cell types.** *[SAFE]* reads label files only. *[OPT-IN: READS IMAGES]* decodes all 1000 development TIFFs (about 1 GB of pixels, tens of seconds) and runs only when `ANALYSE_IMAGES = True`. Example images need `SHOW_IMAGES = True`.

**Prerequisite.** Notebook 00 passes.

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)      # all paths below are relative to the repository root

import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from ct_iqa.data.ldct_iqac import VALID_EXTENSIONS
from ct_iqa.data.splits import load_manifest


def show(title, obj):
    """Print a section title, then the table as plain text (no interactive viewer)."""
    print(f"\n=== {title} ===")
    print(obj.to_string())

TRAIN_IMAGES = Path("data/raw/ldct_iqac/train/image")
labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text(encoding="utf-8"))
manifest = load_manifest("experiments/study_v2/protocol/split_manifest.json", labels)   # raises if altered
PARTS = ("train", "validation", "calibration")
parts = {p: np.array([labels[n] for n in manifest[p]]) for p in PARTS}
print({p: len(v) for p, v in parts.items()}, "| manifest sha256:", manifest["manifest_sha256"][:16] + "...")

## 1. Label frequencies on the 0.2 grid [SAFE]

Training scores are means of five integer 0-4 votes, so they should lie on a 21-level grid {0, 0.2, ..., 4.0}. The table counts images per level overall and per frozen partition; empty levels are shown, not hidden.

In [ ]:
levels = np.round(np.arange(0, 4.0001, 0.2), 1)
idx = np.round(np.array(list(labels.values())) * 5).astype(int)
off_grid = [k for k, v in labels.items() if abs(v * 5 - round(v * 5)) > 1e-6]
print(f"{len(labels)} labels; off the 0.2 grid: {len(off_grid)} {off_grid[:5]}")
freq = pd.DataFrame({p: pd.Series(np.round(v * 5).astype(int)).value_counts().reindex(range(21), fill_value=0) for p, v in parts.items()})
freq.insert(0, "all", freq.sum(axis=1))
freq.index = levels
freq.index.name = "score"
show("Images per score level (empty levels kept)", freq)
print("empty levels overall:", [float(s) for s in freq.index[freq["all"] == 0]] or "none")
print("levels with fewer than 5 images overall:", [float(s) for s in freq.index[freq["all"] < 5]] or "none")

## 2. Descriptive statistics and distribution plots [SAFE]

Summary statistics per partition (raw 0-4 scale) plus shape measures, then three views: counts per grid level, within-partition proportions (so partitions of different size are comparable), and empirical CDFs. Differences between the partitions are *descriptive*; with 100 images per held-out partition, a mean difference of a few tenths is within sampling variation (standard error about 0.1).

In [ ]:
desc = pd.DataFrame({p: pd.Series(v).describe() for p, v in parts.items()}).T
desc["skew"] = [stats.skew(v) for v in parts.values()]
desc["excess kurtosis"] = [stats.kurtosis(v) for v in parts.values()]
desc["SE of mean"] = desc["std"] / np.sqrt(desc["count"])
show("Score statistics per partition (raw 0-4 scale)", desc.round(3))

ks = {f"{a} vs {b}": stats.ks_2samp(parts[a], parts[b]).statistic for a, b in (("train", "validation"), ("train", "calibration"), ("validation", "calibration"))}
show("Two-sample Kolmogorov-Smirnov statistic (descriptive only: scores are heavily tied, so p-values are not used)", pd.DataFrame({"KS statistic": ks}).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
w = 0.27
for j, p in enumerate(PARTS):
    axes[0].bar(levels + (j - 1) * w * 0.2, freq[p].values, width=w * 0.2, label=f"{p} (n={len(parts[p])})")
    axes[1].bar(levels + (j - 1) * w * 0.2, freq[p].values / len(parts[p]), width=w * 0.2, label=p)
    xs = np.sort(parts[p])
    axes[2].step(xs, np.arange(1, len(xs) + 1) / len(xs), where="post", label=p)
axes[0].set(xlabel="score (0-4)", ylabel="images", title="Images per score level")
axes[1].set(xlabel="score (0-4)", ylabel="fraction of partition", title="Within-partition proportions")
axes[2].set(xlabel="score (0-4)", ylabel="cumulative fraction", title="Empirical CDF")
for a in axes:
    a.grid(alpha=0.3)
    a.legend()
fig.tight_layout()
plt.show()

## 3. Image-to-label consistency by filename [SAFE]

No image is opened. Confirms for the development folder: images without labels, labels without images, files that are not `.tif/.tiff`, and that the manifest covers exactly the label keys. Any non-empty list below is a data problem to resolve before training; nothing is dropped automatically.

In [ ]:
files = [p for p in TRAIN_IMAGES.iterdir()]
imgs = [p.name for p in files if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS]
issues = {
    "images without a label": sorted(set(imgs) - set(labels)),
    "labels without an image": sorted(set(labels) - set(imgs)),
    "non-image files / sub-folders": sorted(p.name for p in files if not (p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)),
    "label keys not in manifest": sorted(set(labels) - set().union(*map(set, (manifest[p] for p in PARTS)))),
    "manifest entries not in labels": sorted(set().union(*map(set, (manifest[p] for p in PARTS))) - set(labels)),
}
show("Filename-level consistency", pd.DataFrame({"count": {k: len(v) for k, v in issues.items()}, "examples": {k: v[:3] for k, v in issues.items()}}))
print("CONSISTENT" if not any(issues.values()) else "INCONSISTENT - resolve before training")

## 4. Image metadata and integrity [OPT-IN: READS IMAGES]

Set `ANALYSE_IMAGES = True` to decode every development image (read-only) and record, per file: format, PIL mode, dimensions, dtype after decoding, finiteness, value range, mean, standard deviation, fraction of exact 0 and exact 1 pixels (saturation), and the same mean for the central 224x224 crop the model sees versus the outside. A file that fails to open or decode is recorded with its error, **not skipped silently**.

Expectations from the dataset audit (to be confirmed here, not assumed): TIFF, mode `F`, 512x512, float32 values in [0, 1].

In [ ]:
ANALYSE_IMAGES = False

records = []
if ANALYSE_IMAGES:
    from PIL import Image

    for name in sorted(labels):
        rec = {"filename": name, "partition": next(p for p in PARTS if name in set(manifest[p])), "label": labels[name], "error": ""}
        try:
            with Image.open(TRAIN_IMAGES / name) as im:
                rec.update(format=im.format, mode=im.mode, width=im.size[0], height=im.size[1])
                a = np.array(im, dtype=np.float64)
            fin = np.isfinite(a)
            v = a[fin]
            c = a[144:368, 144:368] if a.shape == (512, 512) else a
            rec.update(finite_fraction=float(fin.mean()), vmin=float(v.min()) if v.size else np.nan, vmax=float(v.max()) if v.size else np.nan,
                       mean=float(v.mean()) if v.size else np.nan, std=float(v.std()) if v.size else np.nan,
                       frac_zero=float((a == 0).mean()), frac_one=float((a == 1).mean()), crop_mean=float(np.nanmean(c)))
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
        records.append(rec)
    meta = pd.DataFrame(records).set_index("filename")
    print(f"analysed {len(meta)} images; unreadable: {(meta['error'] != '').sum()}")
else:
    print("ANALYSE_IMAGES is False: no image was opened.")

In [ ]:
if records:
    ok = meta[meta["error"] == ""]
    show("Format / mode / dimensions (counts of distinct combinations)", ok.groupby(["format", "mode", "width", "height"]).size().rename("images").to_frame())
    show("Pixel statistics across images", ok[["vmin", "vmax", "mean", "std", "frac_zero", "frac_one", "finite_fraction"]].describe().T.round(4))
    flags = pd.DataFrame({
        "unreadable": meta["error"] != "",
        "not 512x512": (meta["width"] != 512) | (meta["height"] != 512),
        "mode not F": meta["mode"] != "F",
        "format not TIFF": meta["format"] != "TIFF",
        "non-finite pixels": meta["finite_fraction"] < 1,
        "values outside [0, 1]": (meta["vmin"] < 0) | (meta["vmax"] > 1),
        "constant image (std = 0)": meta["std"] == 0,
        "heavy saturation (>5% exact 0 or 1)": (meta["frac_zero"] > 0.05) | (meta["frac_one"] > 0.05),
    })
    show("Flagged images by criterion (images are kept; this is a report, not a filter)", pd.DataFrame({"images": flags.sum()}))
    flagged = flags.any(axis=1)
    print(f"{int(flagged.sum())} of {len(meta)} images carry at least one flag.")
    if flagged.any():
        show("Flagged images", flags[flagged].assign(partition=meta.loc[flagged, "partition"], label=meta.loc[flagged, "label"], error=meta.loc[flagged, "error"]))

In [ ]:
if records:
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
    axes[0].hist(ok["mean"], bins=30); axes[0].set(xlabel="per-image mean intensity", ylabel="images", title="Mean intensity")
    axes[1].hist(ok["std"], bins=30); axes[1].set(xlabel="per-image std", ylabel="images", title="Intensity spread")
    axes[2].scatter(ok["label"], ok["std"], s=6, alpha=0.5); axes[2].set(xlabel="score", ylabel="per-image std", title="Spread vs score")
    axes[3].scatter(ok["mean"], ok["crop_mean"], s=6, alpha=0.5)
    axes[3].plot([ok["mean"].min(), ok["mean"].max()], [ok["mean"].min(), ok["mean"].max()], "k--", lw=1)
    axes[3].set(xlabel="mean, whole image", ylabel="mean, central 224x224 crop", title="Crop vs whole image")
    for a in axes: a.grid(alpha=0.3)
    fig.tight_layout(); plt.show()
    show("Mean intensity by partition (descriptive)", ok.groupby("partition")[["mean", "std", "crop_mean"]].agg(["mean", "std"]).round(4))
    rho = stats.spearmanr(ok["label"], ok["std"])[0]
    print(f"Spearman(score, per-image std) = {rho:.3f} (n = {len(ok)}); a descriptive hint that intensity spread relates to label, not a finding.")

## 5. Representative images [OPT-IN: READS IMAGES]

Set `SHOW_IMAGES = True` to display one training-partition image for each of four score levels (lowest, 25th, 75th percentile, highest). The dashed box is the 224x224 centre crop that the model receives; everything outside it is discarded by the preprocessing. Windowing is the file's own [0, 1] range.

In [ ]:
SHOW_IMAGES = False

if SHOW_IMAGES:
    from PIL import Image
    import matplotlib.patches as mpatches

    tr = sorted(manifest["train"], key=lambda n: (labels[n], n))
    picks = [tr[0], tr[len(tr) // 4], tr[3 * len(tr) // 4], tr[-1]]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
    for ax, n in zip(axes, picks):
        with Image.open(TRAIN_IMAGES / n) as im:
            arr = np.array(im, dtype=np.float32)
        ax.imshow(arr, cmap="gray", vmin=0, vmax=1)
        ax.add_patch(mpatches.Rectangle((144, 144), 224, 224, fill=False, ec="tab:red", ls="--", lw=1.2))
        ax.set_title(f"{n}  score {labels[n]:.1f}", fontsize=9); ax.axis("off")
    fig.tight_layout(); plt.show()
else:
    print("SHOW_IMAGES is False: no image was opened.")

## 6. Findings and limitations

*Findings are whatever the tables above show when run; none are asserted in advance.* Points to read off:

- whether any score level is empty or very sparse (relevant to the pair threshold in Experiment B and to calibration coverage in C);
- whether train / validation / calibration look alike (they are one random image-level split, so large differences would be surprising);
- (opt-in) whether any file is unreadable, non-512x512, non-finite, outside [0, 1], constant or saturated.

**Limitations.** Labels are means of five raters, so they are not ground truth quality; the split is image-level and patient identity is unavailable, so images from the same patient may appear in several partitions; flagged images are reported but never removed, so any decision to exclude data would be a protocol change needing approval; the official test set was not examined.